In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

In [3]:
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [ ]:
snapshot_download(
    repo_id="thainamhoang/era5-climate-learn",
    repo_type="dataset",
    allow_patterns="data/5.625deg/2m_temperature/**",
    local_dir="/workspace"
)

snapshot_download(
    repo_id="thainamhoang/era5-climate-learn",
    repo_type="dataset",
    allow_patterns="data/1.40625deg/2m_temperature/**",
    local_dir="/workspace"
)

snapshot_download(
    repo_id="thainamhoang/era5-climate-learn",
    repo_type="dataset",
    allow_patterns="data/1.40625deg/constants.npz",
    local_dir="/workspace"
)

snapshot_download(
    repo_id="thainamhoang/era5-climate-learn",
    repo_type="dataset",
    allow_patterns="data/5.625deg/constants.npz",
    local_dir="/workspace"
)

In [1]:
import xarray as xr
import numpy as np

In [2]:
!pwd

/Users/thainamhoang/Desktop/ph-reg/ssl_training_pipeline


In [11]:
ds = xr.open_dataset("/Users/thainamhoang/Desktop/ph-reg/data/cerra/constant.nc")

# Check what variables are inside
print(ds)
print(ds.data_vars)
print(ds.coords)

<xarray.Dataset> Size: 27MB
Dimensions:     (valid_time: 1, y: 1069, x: 1069)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8B 2010-01-01
    latitude    (y, x) float64 9MB ...
    longitude   (y, x) float64 9MB ...
    expver      <U4 16B ...
Dimensions without coordinates: y, x
Data variables:
    lsm         (valid_time, y, x) float32 5MB ...
    orog        (valid_time, y, x) float32 5MB ...
Attributes:
    GRIB_centre:             eswi
    GRIB_centreDescription:  Norrkoping
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Norrkoping
    history:                 2026-03-29T09:19 GRIB to CDM+CF via cfgrib-0.9.1...
Data variables:
    lsm      (valid_time, y, x) float32 5MB ...
    orog     (valid_time, y, x) float32 5MB ...
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8B 2010-01-01
    latitude    (y, x) float64 9MB ...
    longitude   (y, x) float64 9MB ...
    expver      <U4 16B ...


In [1]:
import numpy as np
import xarray as xr

ds = xr.open_dataset("/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/constant.nc")

# Orography — ERA5 stores geopotential, convert to metres
# If variable is named 'z': divide by gravity constant
if "z" in ds:
    oro = (ds["z"].values / 9.80665).astype(np.float32)
elif "orog" in ds:
    oro = ds["orog"].values.astype(np.float32)

# Land-sea mask
lsm = ds["lsm"].values.astype(np.float32)

# Squeeze out any extra dimensions — ERA5 constants sometimes have
# a time dimension of length 1: [1, H, W] → [H, W]
oro = oro.squeeze()
lsm = lsm.squeeze()

print(f"Orography shape: {oro.shape}, range: [{oro.min():.1f}, {oro.max():.1f}] m")
print(f"LSM shape      : {lsm.shape}, unique values: {np.unique(lsm)}")

# Save
np.savez("constants.npz", orography=oro, lsm=lsm)
print("Saved constants.npz")

Orography shape: (1069, 1069), range: [-411.0, 4001.4] m
LSM shape      : (1069, 1069), unique values: [0.0000000e+00 1.5258789e-04 1.8310547e-04 ... 9.8989868e-01 9.9038696e-01
 1.0000000e+00]
Saved constants.npz


In [3]:
xxx = np.load("/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/constant.npz")
xxx["orography"]

array([[ 9.83273625e-01, -1.38739705e+00,  2.32746616e-01, ...,
         3.23363190e+02,  3.23735077e+02,  3.25983978e+02],
       [ 4.83039647e-01,  1.10143855e-01,  6.06288016e-01, ...,
         3.24736755e+02,  3.25872772e+02,  3.26864777e+02],
       [-5.13683915e-01, -1.49131054e-02, -5.13040721e-01, ...,
         3.32732269e+02,  3.36384949e+02,  3.30879456e+02],
       ...,
       [-2.65908241e-01,  1.10243344e+00, -2.12693477e+00, ...,
         1.01118103e+02,  1.02225433e+02,  9.92240295e+01],
       [-2.01116896e+00,  1.73040390e+00, -6.38094425e-01, ...,
         1.05113663e+02,  1.04482330e+02,  1.01857162e+02],
       [-2.66140819e-01, -1.40031055e-01, -5.14852405e-01, ...,
         1.11357285e+02,  1.09110611e+02,  1.04610435e+02]],
      shape=(1068, 1068), dtype=float32)

In [4]:
yyy = np.load("/Users/thainamhoang/Desktop/ph-reg/data/5.625deg/constants.npz")
yyy["orography"]

array([[ 2.59010498e+03,  2.64685010e+03,  2.70220239e+03, ...,
         2.44911182e+03,  2.50664014e+03,  2.54850293e+03],
       [ 2.15320410e+03,  2.38679810e+03,  2.51151514e+03, ...,
         1.67626062e+03,  1.89645618e+03,  1.99462939e+03],
       [ 2.78775488e+03,  3.06564844e+03,  3.28496899e+03, ...,
         1.11872510e+03,  2.12768921e+03,  2.52195361e+03],
       ...,
       [-1.43596753e-01,  3.78607571e-01, -4.91735637e-01, ...,
         4.72914553e+00,  1.17506355e-01,  2.04495415e-01],
       [ 3.04719880e-02,  5.52678883e-01, -9.26906586e-01, ...,
         1.36140335e+02,  3.68541794e+01, -2.05799007e+00],
       [ 3.04719880e-02,  2.04540491e-01,  4.65645254e-01, ...,
         1.16184330e+00, -4.91643548e-01, -2.30661318e-01]],
      shape=(32, 64), dtype=float32)

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import glob
import os

api = HfApi()

# api.upload_file(
#     path_or_fileobj="/Users/thainamhoang/Desktop/ph-reg/data/1.40625deg/constants.npz",
#     repo_id="thainamhoang/era5-climate-learn",
#     repo_type="dataset",
#     path_in_repo="data/1.40625deg/constants.npz"  # where it lands in the HF repo
# )
# Define local pattern and repo destination

source_dir = "/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature/train"

for file in sorted(os.listdir(source_dir)):
    for year in range(2015, 2020):
        if f"{year}" in file:
    # if "2014" in file:
            api.upload_file(
                path_or_fileobj=f"{source_dir}/{file}",
                path_in_repo=f"data/5.5km/2m_temperature/train/{file}",
                repo_id="thainamhoang/cerra_climate_learn",
                repo_type="dataset",
            )

# api.upload_folder(
#     folder_path="/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature/train",
#     repo_id="thainamhoang/cerra_climate_learn",
#     repo_type="dataset",
#     path_in_repo="data/5.5km/2m_temperature/train",
# )

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path
import glob
import os

api = HfApi()
source_dir = "/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature/"


# api.upload_file(
#     path_or_fileobj=f"{source_dir}/climatology.npz",
#     repo_id="thainamhoang/era5-climate-learn",
#     repo_type="dataset",
#     path_in_repo="data/5.5km/climatology.npz"
# )

api.upload_file(
    path_or_fileobj=f"{source_dir}/lat.npy",
    repo_id="thainamhoang/cerra-climate-learn",
    repo_type="dataset",
    path_in_repo="data/5.5km/lat.npy"
)

api.upload_file(
    path_or_fileobj=f"{source_dir}/lon.npy",
    repo_id="thainamhoang/cerra-climate-learn",
    repo_type="dataset",
    path_in_repo="data/5.5km/lon.npy"
)




# api.upload_folder(
#     folder_path="/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature/train",
#     repo_id="thainamhoang/cerra_climate_learn",
#     repo_type="dataset",
#     path_in_repo="data/5.5km/2m_temperature/train",
# )

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/thainamhoang/era5-climate-learn/commit/efd30071a6bc72a16d97da793c8138d54916b693', commit_message='Upload data/5.5km/lon.npy with huggingface_hub', commit_description='', oid='efd30071a6bc72a16d97da793c8138d54916b693', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/thainamhoang/era5-climate-learn', endpoint='https://huggingface.co', repo_type='dataset', repo_id='thainamhoang/era5-climate-learn'), pr_revision=None, pr_num=None)

In [7]:
from huggingface_hub import HfApi
from pathlib import Path
import glob
import os

api = HfApi()

api.upload_folder(
    folder_path="/Users/thainamhoang/Desktop/ph-reg/data/cerra/44km/2m_temperature/",
    repo_id="thainamhoang/cerra-climate-learn",
    repo_type="dataset",
    path_in_repo="data/44km/2m_temperature/",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/thainamhoang/cerra-climate-learn/commit/4fd48fcc2a98aeef9eab114e9de23bf397523c8d', commit_message='Upload folder using huggingface_hub', commit_description='', oid='4fd48fcc2a98aeef9eab114e9de23bf397523c8d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/thainamhoang/cerra-climate-learn', endpoint='https://huggingface.co', repo_type='dataset', repo_id='thainamhoang/cerra-climate-learn'), pr_revision=None, pr_num=None)